# SE-AT-CNN ablation — architecture and chronology gate

**IMPORTANT:** CIC provided CSVs cannot support genuine temporal attention because timestamps/groups are missing. This notebook validates A0–A3 architectures on artificial tensors only, not a training result. Use notebook 08 to train on CTU temporal flows when pyarrow is installed.

In [ ]:
from pathlib import Path
import sys, os, json
# On Kaggle upload the project folder or add it as a Kaggle Dataset.
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT / 'network-intrusion-detection-se-at-cnn' / 'src').exists(): ROOT = ROOT / 'network-intrusion-detection-se-at-cnn'
sys.path.insert(0, str(ROOT / 'src'))
CIC_ZIP = Path(os.getenv('CIC_ZIP', '/kaggle/input/cic-ids2017/CIC-IDS2017.zip'))
CTU_ZIP = Path(os.getenv('CTU_ZIP', '/kaggle/input/ctu-13-dataset/ctu-13-dataset.zip'))
OUTPUT = Path('/kaggle/working/nids_outputs' if Path('/kaggle/working').exists() else 'reports/notebook_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Project root:',ROOT, '\nCIC:',CIC_ZIP,'exists:',CIC_ZIP.exists(), '\nCTU:',CTU_ZIP,'exists:',CTU_ZIP.exists())

## Check PyTorch architectures

In [ ]:
import torch
from nids.models import SEATCNN
torch.manual_seed(42)
x=torch.randn(6,16,8)
for se,att in [(False,False),(True,False),(False,True),(True,True)]:
 net=SEATCNN(features=8,classes=4,use_se=se,use_attention=att)
 output=net(x)
 assert output.shape==(6,4)
 print('SE:',se,'Attention:',att,'parameters:',sum(p.numel() for p in net.parameters()))

## Confirm CIC limitation

In [ ]:
from nids.datasets import cic_chunks
first=next(cic_chunks(CIC_ZIP,200))
assert not any(k.lower() in {'timestamp','starttime','srcaddr','source ip'} for k in first.columns)
print('Temporal training on CIC CSV explicitly BLOCKED: missing original times and hosts')